In [1]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import  numpy as np

In [3]:
# load data
DATA_PATH = library_path.parent / "data"
PLOTS_PATH = library_path.parent / "plots"

In [4]:
df = pd.read_csv(
    DATA_PATH / "Data_verysmall_Apr2026.csv", 
    encoding='latin-1',
    low_memory=False
)

In [5]:
df.columns

Index(['ID', 'Sex', 'wt_int', 'eqv3', 'eqv5', 'MO5L', 'SC5L', 'UA5L', 'PD5L',
       'AD5L', 'EQvas', 'FULLHEALTH', 'anyprob', 'mo2cat', 'sc2cat', 'ua2cat',
       'pd2cat', 'ad2cat', 'LSS', 'LSS_rs', 'bmi_calc', 'bmi_cat', 'edu_cat',
       'edu_pst', 'christian_bin', 'EQ_index', 'emp_cat', 'age3cat', 'age7cat',
       'eth4cat', 'srh', 'paVig', 'paMod', 'smoke_ever', 'smoke_ecig',
       'alcohol_yr', 'diabetes', 'obese', 'resp', 'skin', 'dataset',
       'SurveyWeight', 'daphnie_surveywt', 'sat', 'ill_dis', 'meds_num',
       'svy_wt', 'eth_oth', 'eth_oth2', 'emp_daphniepilot', 'emp_cat_daphnie',
       'eth4cat_hse2', 'emp_cat_hse2019', 'emp_cat_hse', 'paVig_daphnie',
       'paMod_daphnie', '_merge', 'PA_vig', 'PA_mod', 'edu_daphniepilot',
       'edu_cat_daphniepilot', 'edu_pst_daphniepilot', 'srh_2', 'edu_cat_2',
       'white'],
      dtype='str')

In [6]:
eth4cat_pct = (
    df.groupby("dataset")['eth4cat']
    .value_counts(normalize=True, dropna=False)
    .mul(100).round(1)
    .rename("pct")
    .reset_index()
    .pivot(index="eth4cat", columns="dataset", values="pct")
    .fillna(0)
)

eth4cat_n = pd.crosstab(
    df["eth4cat"].fillna("(missing)"),
    df["dataset"],
    margins=True,
    margins_name="Total",
)

# column-wise proportions (% within each dataset)
eth4cat_pct = eth4cat_n.div(eth4cat_n.loc["Total"]).mul(100).round(1)

print("Counts:")
display(eth4cat_n)

print("\nColumn % (within each dataset):")
display(eth4cat_pct)

Counts:


dataset,DAPHNIE 2024,DAPHNIE Pilot 2023,HSE 2017,HSE 2018,HSE 2019,HSE 2022,Total
eth4cat,,,,,,,
(missing),32,139,30,26,29,66,322
Asian,480,126,534,573,678,487,2878
Black,477,78,210,249,233,203,1450
Others,167,129,143,197,198,196,1030
White,4081,1185,6923,6945,6888,6638,32660
Total,5237,1657,7840,7990,8026,7590,38340



Column % (within each dataset):


dataset,DAPHNIE 2024,DAPHNIE Pilot 2023,HSE 2017,HSE 2018,HSE 2019,HSE 2022,Total
eth4cat,,,,,,,
(missing),0.6,8.4,0.4,0.3,0.4,0.9,0.8
Asian,9.2,7.6,6.8,7.2,8.4,6.4,7.5
Black,9.1,4.7,2.7,3.1,2.9,2.7,3.8
Others,3.2,7.8,1.8,2.5,2.5,2.6,2.7
White,77.9,71.5,88.3,86.9,85.8,87.5,85.2
Total,100.0,100.0,100.0,100.0,100.0,100.0,100.0


In [7]:
smoke_pct = (
    df.groupby("dataset")["smoke_ever"]
    .value_counts(normalize=True, dropna=False)
    .mul(100).round(1)
    .rename("pct")
    .reset_index()
    .pivot(index="smoke_ever", columns="dataset", values="pct")
    .fillna(0)
)

smoke_n = pd.crosstab(
    df["smoke_ever"].fillna("(missing)"),
    df["dataset"],
    margins=True,
    margins_name="Total",
)

# column-wise proportions (% within each dataset)
smoke_pct = smoke_n.div(smoke_n.loc["Total"]).mul(100).round(1)

print("Counts:")
display(smoke_n)

print("\nColumn % (within each dataset):")
display(smoke_pct)

Counts:


dataset,DAPHNIE 2024,DAPHNIE Pilot 2023,HSE 2017,HSE 2018,HSE 2019,HSE 2022,Total
smoke_ever,,,,,,,
(missing),541,0,37,37,29,54,698
no,2585,1365,3450,3531,3715,3627,18273
yes,2111,292,4353,4422,4282,3909,19369
Total,5237,1657,7840,7990,8026,7590,38340



Column % (within each dataset):


dataset,DAPHNIE 2024,DAPHNIE Pilot 2023,HSE 2017,HSE 2018,HSE 2019,HSE 2022,Total
smoke_ever,,,,,,,
(missing),10.3,0.0,0.5,0.5,0.4,0.7,1.8
no,49.4,82.4,44.0,44.2,46.3,47.8,47.7
yes,40.3,17.6,55.5,55.3,53.4,51.5,50.5
Total,100.0,100.0,100.0,100.0,100.0,100.0,100.0


# Column Exploration

Reference overview of all columns: dtypes, missingness, unique counts, and value distributions.
Used to inform encoding decisions before wrangling.

In [8]:
CATEGORICAL_THRESHOLD = 30  # columns with <= this many unique values get value listings

overview_rows = []
for col in df.columns:
    n_missing = df[col].isna().sum()
    n_unique = df[col].nunique(dropna=True)
    overview_rows.append({
        "column": col,
        "dtype": str(df[col].dtype),
        "missing": n_missing,
        "missing_pct": round(n_missing / len(df) * 100, 1),
        "n_unique": n_unique,
    })

overview = pd.DataFrame(overview_rows).set_index("column")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
display(overview)

Shape: 38,340 rows × 65 columns



,dtype,missing,missing_pct,n_unique
column,,,,
ID,float64,30500,79.6,7840
Sex,str,50,0.1,2
wt_int,float64,6894,18.0,20463
eqv3,str,6894,18.0,5
eqv5,str,6894,18.0,7
...,...,...,...,...
edu_cat_daphniepilot,str,36684,95.7,3
edu_pst_daphniepilot,str,36684,95.7,3
srh_2,str,324,0.8,4


In [9]:
# Categorical columns: list all unique values (sorted)
cat_cols = overview[overview["n_unique"] <= CATEGORICAL_THRESHOLD].index.tolist()

print(f"{'='*60}")
print(f"CATEGORICAL COLUMNS ({len(cat_cols)} columns, <= {CATEGORICAL_THRESHOLD} unique values)")
print(f"{'='*60}\n")

for col in cat_cols:
    vals = sorted(df[col].dropna().unique(), key=str)
    missing_pct = overview.loc[col, "missing_pct"]
    print(f"  {col}  [missing: {missing_pct}%]")
    print(f"    {vals}\n")

CATEGORICAL COLUMNS (58 columns, <= 30 unique values)

  Sex  [missing: 0.1%]
    ['Female', 'Male']

  eqv3  [missing: 18.0%]
    ['Age of household member refused', 'Highest Tertile (>£38,897)', 'Item not applicable', 'Lowest Tertile (<=£19,204)', 'Middle Tertile (>£19,204- £38,897)']

  eqv5  [missing: 18.0%]
    ['Age of household member refused', 'Highest Quintile (>£52,816)', 'Item not applicable', 'Lowest Quintile (<=£14,832)', 'Middle Quintile (>£22,100 <=£32,231 )', 'Second highest Quintile (>£32,231 <=£52,816)', 'Second lowest Quintile (>£14,832<=£22,100)']

  MO5L  [missing: 44.8%]
    ['Extreme/Unable', 'Moderate', 'No Problems', 'Severe', 'Slight']

  SC5L  [missing: 44.9%]
    ['Extreme/Unable', 'Moderate', 'No Problems', 'Severe', 'Slight']

  UA5L  [missing: 44.9%]
    ['Extreme/Unable', 'Moderate', 'No Problems', 'Severe', 'Slight']

  PD5L  [missing: 44.9%]
    ['Extreme/Unable', 'Moderate', 'No Problems', 'Severe', 'Slight']

  AD5L  [missing: 45.0%]
    ['Extreme/Un

In [10]:
# Continuous columns: summary statistics
cont_cols = overview[overview["n_unique"] > CATEGORICAL_THRESHOLD].index.tolist()

print(f"{'='*60}")
print(f"CONTINUOUS COLUMNS ({len(cont_cols)} columns, > {CATEGORICAL_THRESHOLD} unique values)")
print(f"{'='*60}\n")

if cont_cols:
    display(df[cont_cols].describe().T.round(3))

CONTINUOUS COLUMNS (7 columns, > 30 unique values)



,count,mean,std,min,25%,50%,75%,max
ID,7840.0,2705978.075,3474.808,2700001.000,2702943.750,2705939.500,2709022.250,2711995.000
wt_int,31446.0,0.989,0.522,0.221,0.750,0.867,1.091,14.115
EQvas,20563.0,75.501,20.251,0.000,68.000,80.000,90.000,100.000
bmi_calc,25080.0,27.573,12.324,0.133,23.739,26.959,30.908,1076.391
EQ_index,20894.0,0.863,0.228,-0.567,0.853,0.944,1.000,1.000
svy_wt,37531.0,0.989,0.482,0.221,0.772,0.900,1.052,14.115


# Variable Encoding and Wrangling

Encoding strategy:
- **Ordinal**: EQ-5D-5L dimensions, income (eqv3/eqv5), education, self-rated health, alcohol frequency, medication count, physical activity (`paVig`: 1–3, `paMod`: 1–4) — all have a clear rank order
- **Binary**: Sex, ill_dis, and binary health condition flags
- **Dummy**: ethnicity (eth4cat) and employment status (emp_cat) — nominal categories with no defensible linear order; ordinal encoding would impose false distances in clustering and density ratio estimation

In [11]:
def ordinal_encode(series, mapping, col_name):
    """Map string series to ordinal integers; print any values not covered by the mapping."""
    cleaned = series.str.strip() if series.dtype == object else series
    encoded = cleaned.map(mapping)
    unmapped = cleaned[cleaned.notna() & encoded.isna()].unique()
    if len(unmapped) > 0:
        print(f"  WARNING {col_name}: unmapped values -> {sorted(unmapped, key=str)}")
    return encoded


# Working copy — keeps df intact for reference
df_w = df.copy()

# Drop columns not needed for analysis
df_w = df_w.drop(columns=["ID"])

# Drop rows with invalid age7cat sentinel codes
n_before = len(df_w)
df_w = df_w[~df_w["age7cat"].isin([-1, 0])].copy()
print(f"Dropped {n_before - len(df_w):,} rows with invalid age7cat (-1 or 0).")
print(f"Rows retained: {len(df_w):,}")

# Harmonise dataset label for the DAPHNIE pilot wave
df_w["dataset"] = df_w["dataset"].str.replace("DAPHNIE Pilot 2023", "DAPHNIE 2023", regex=False)
print("\ndataset value counts:")
print(df_w["dataset"].value_counts())

# BMI: values above 70 are physiologically implausible → NaN
n_bmi_outliers = (df_w["bmi_calc"] > 70).sum()
df_w.loc[df_w["bmi_calc"] > 70, "bmi_calc"] = np.nan
print(f"\nReplaced {n_bmi_outliers:,} implausible bmi_calc values (> 70) with NaN.")

Dropped 0 rows with invalid age7cat (-1 or 0).
Rows retained: 38,340

dataset value counts:
dataset
HSE 2019        8026
HSE 2018        7990
HSE 2017        7840
HSE 2022        7590
DAPHNIE 2024    5237
DAPHNIE 2023    1657
Name: count, dtype: int64

Replaced 30 implausible bmi_calc values (> 70) with NaN.


In [12]:
# Drop derived EQ-5D-5L columns — redundant with the 5 encoded dimensions
eq5d_derived = ["FULLHEALTH", "anyprob", "mo2cat", "sc2cat", "ua2cat", "pd2cat", "ad2cat"]

# Drop DAPHNIE-specific columns — present only in DAPHNIE waves, not harmonised across datasets
daphnie_specific = [
    "SAH", "age", "age10", "educ", "inco3", "inco5",
    "empl", "hpempl", "mars", "size", "limc", "scls",
    "urbn", "help", "daphnie_surveywt",
]

cols_to_drop = [c for c in eq5d_derived + daphnie_specific if c in df_w.columns]
df_w = df_w.drop(columns=cols_to_drop)
n_eq5d_dropped = sum(c in cols_to_drop for c in eq5d_derived)
n_daphnie_dropped = sum(c in cols_to_drop for c in daphnie_specific)

print(
    f"Dropped {len(cols_to_drop)} columns "
    f"({n_eq5d_dropped} EQ-5D-5L derived, {n_daphnie_dropped} DAPHNIE-specific)."
)
print(f"Remaining: {df_w.shape[1]} columns")

Dropped 8 columns (7 EQ-5D-5L derived, 1 DAPHNIE-specific).
Remaining: 56 columns


In [13]:
# EQ-5D-5L dimensions: ordinal 1–5
eq5d_map = {
    "No Problems":       1,
    "Slight":            2,
    "Moderate":          3,
    "Severe":            4,
    "Extreme/Unable":    5,
}
eq5d_dims = ["MO5L", "SC5L", "UA5L", "PD5L", "AD5L"]

print("EQ-5D-5L encoding:")
for col in eq5d_dims:
    df_w[col] = ordinal_encode(df_w[col], eq5d_map, col)
print("  Done.")

EQ-5D-5L encoding:
  Done.


In [14]:
# Income: ordinal lowest→highest
# Stata non-response codes that survived as strings — replace with NaN before mapping
income_na_codes = {"Age of household member refused", "Item not applicable"}
for col in ["eqv3", "eqv5"]:
    df_w[col] = df_w[col].str.strip().replace(income_na_codes, np.nan)
    
eqv3_map = {
    "Lowest Tertile (<=£19,204)":           1,
    "Middle Tertile (>£19,204- £38,897)":   2,
    "Highest Tertile (>£38,897)":           3,
}
eqv5_map = {
    "Lowest Quintile (<=£14,832)":              1,
    "Second lowest Quintile (>£14,832<=£22,100)":        2,
    "Middle Quintile (>£22,100 <=£32,231 )":    3,
    "Second highest Quintile (>£32,231 <=£52,816)":        4,
    "Highest Quintile (>£52,816)":              5,
}

print("Income encoding:")
df_w["eqv3"] = ordinal_encode(df_w["eqv3"], eqv3_map, "eqv3")
df_w["eqv5"] = ordinal_encode(df_w["eqv5"], eqv5_map, "eqv5")
print("  Done.")

Income encoding:
  Done.


In [15]:
df['age7cat'].value_counts(dropna=False)

age7cat
45-54      6565
55-64      6509
35-44      6457
65-74      6321
25-34      5380
75+        4490
18-24      2551
NaN          67
Name: count, dtype: int64

In [16]:
# Other ordinal variables

# -9 is a Stata sentinel code in edu_cat — replace before mapping
df_w["edu_cat"] = df_w["edu_cat"].str.strip().replace({"-9": np.nan})

edu_cat_map = {
    "No qualification":      1,
    "Below degree":          2,
    "Degree or equivalent":  3,
    "Higher than degree":    4,
}
edu_pst_map = {
    "Primary/None": 1,
    "Secondary":    2,
    "Tertiary":     3,
}
srh_map = {
    "Poor/Very bad":        1,
    "Fair/bad":             2,
    "Good/fair":            3,
    "Very good/Good":       4,
    "Very good/Excellent":  5,
}
srh_2_map = {
    "poor/bad/v.bad":         1,
    "fair":                   2,
    "Good":                   3,
    "v.good/Excellent":       4
}
alcohol_map = {
    "never":            0,
    "monthly or less":  1,
    "monthly+":         2,
    "weekly+":          3,
    "4+ per wk":        4,
}
meds_map = {
    "none":     0,
    "1 meds":   1,
    "2 meds":   2,
    "3+ meds":  3,
}
ill_dis_map = {
    "no":  0,
    "yes": 1,
}
bmi_cat_map = {
    "Underweight": 1,
    "Normal":      2,
    "Overweight":  3,
    "Obese":       4,
}
age3cat_map = {
    "16-34": 1,
    "35-54": 2,
    "55+":   3,
}
age7cat_map = {
    "18-24": 1,
    "25-34": 2,
    "35-44": 3,
    "45-54": 4,
    "55-64": 5,
    "65-74": 6,
    "75+":   7,
}
paVig_map = {
    "<2":       1,
    "1-2 days": 2,
    "3+ days":  3,
}
paMod_map = {
    "<2":       1,
    "1-2 days": 2,
    "3-4 days": 3,
    "5+ days":  4,
}

print("Other ordinal encoding:")
for col, mapping in [
    ("edu_cat",   edu_cat_map),
    ("edu_pst",   edu_pst_map),
    ("srh",       srh_map),
    ("srh_2",     srh_2_map),
    ("meds_num",  meds_map),
    ("ill_dis",   ill_dis_map),
    ("bmi_cat",   bmi_cat_map),
    ("age3cat",   age3cat_map),
    ("age7cat",   age7cat_map),
    ("paVig",     paVig_map),
    ("paMod",     paMod_map),
]:
    if col in df_w.columns:
        df_w[col] = ordinal_encode(df_w[col], mapping, col)

# alcohol_yr: normalise case before mapping
if "alcohol_yr" in df_w.columns:
    df_w["alcohol_yr"] = ordinal_encode(
        df_w["alcohol_yr"].str.strip().str.lower(),
        alcohol_map,
        "alcohol_yr",
    )

print("  Done.")

Other ordinal encoding:
  WARNING age7cat: unmapped values -> ['45-54 ', '55-64  ', '65-74 ']
  Done.


In [17]:
# Binary encoding
df_w["Sex"] = df_w["Sex"].str.strip().map({"Male": 0, "Female": 1})

for col in ["smoke_ever", "smoke_ecig"]:
    if col in df_w.columns:
        df_w[col] = df_w[col].str.strip().map({"no": 0, "yes": 1})

if "christian_bin" in df_w.columns:
    df_w["christian_bin"] = df_w["christian_bin"].str.strip().map({"Non-Christian": 0, "Christian": 1})

for col in ["obese", "resp", "skin", "diabetes", "bowel", "mus"]:
    if col in df_w.columns:
        df_w[col] = pd.to_numeric(df_w[col], errors="coerce")

print("Binary encoding done.")
print(df_w[["Sex", "smoke_ever", "smoke_ecig", "christian_bin"]].dtypes)

Binary encoding done.
Sex              float64
smoke_ever       float64
smoke_ecig         int64
christian_bin    float64
dtype: object


In [18]:
# Binary recoding: eth4cat → eth2cat (White vs Non-White)
# Must run before the dummy-encoding cell, while eth4cat still holds string values.
df_w['eth2cat'] = df_w['eth4cat'].map({
    'White':  0,
    'Asian':  1,
    'Black':  1,
    'Others': 1,
})
# NaN propagated where eth4cat is missing (~0.8% of rows)

# Binary encoding: edu_cat_2 (pre-existing column, still string at this point)
df_w['edu_cat_2'] = df_w['edu_cat_2'].str.strip().map({
    'below degree':         0,
    'Degree or equivalent': 1,
})

print("eth2cat (0 = White, 1 = Non-White) — column % by dataset:")
eth2_n = pd.crosstab(df_w['eth2cat'].fillna('(missing)'), df_w['dataset'],
                     margins=True, margins_name='Total')
print((eth2_n.div(eth2_n.loc['Total']).mul(100).round(1)).to_string())

print("\nedu_cat_2 (0 = below degree, 1 = Degree or equivalent) — column % by dataset:")
edu2_n = pd.crosstab(df_w['edu_cat_2'].fillna('(missing)'), df_w['dataset'],
                     margins=True, margins_name='Total')
print((edu2_n.div(edu2_n.loc['Total']).mul(100).round(1)).to_string())

eth2cat (0 = White, 1 = Non-White) — column % by dataset:
dataset    DAPHNIE 2023  DAPHNIE 2024  HSE 2017  HSE 2018  HSE 2019  HSE 2022  Total
eth2cat                                                                             
0.0                71.5          77.9      88.3      86.9      85.8      87.5   85.2
1.0                20.1          21.5      11.3      12.8      13.8      11.7   14.0
(missing)           8.4           0.6       0.4       0.3       0.4       0.9    0.8
Total             100.0         100.0     100.0     100.0     100.0     100.0  100.0

edu_cat_2 (0 = below degree, 1 = Degree or equivalent) — column % by dataset:
dataset    DAPHNIE 2023  DAPHNIE 2024  HSE 2017  HSE 2018  HSE 2019  HSE 2022  Total
edu_cat_2                                                                           
0.0                55.5          58.0      70.7      71.1      69.9      64.5   67.0
1.0                44.5          41.4      28.8      28.4      29.5      34.6   32.4
(missing)    

In [19]:
# Dummy encoding for nominal categoricals
dummy_cols = [c for c in ["eth4cat", "emp_cat"] if c in df_w.columns]

df_w = pd.get_dummies(df_w, columns=dummy_cols, drop_first=False, dtype=float)

new_dummies = [c for c in df_w.columns if any(c.startswith(d + "_") for d in dummy_cols)]
print(f"Dummy columns created ({len(new_dummies)}):")
print(new_dummies)

Dummy columns created (13):
['emp_cat_daphnie', 'eth4cat_hse2', 'emp_cat_hse2019', 'emp_cat_hse', 'eth4cat_Asian', 'eth4cat_Black', 'eth4cat_Others', 'eth4cat_White', 'emp_cat_Employed', 'emp_cat_Other (Sick/Home/etc)', 'emp_cat_Retired', 'emp_cat_Student', 'emp_cat_Unemployed']


In [20]:
# Final check
print(f"Final shape: {df_w.shape[0]:,} rows × {df_w.shape[1]} columns")
print(f"\nRemaining object/string columns (may need attention):")
obj_cols = df_w.select_dtypes(include=["object", "string"]).columns.tolist()
for col in obj_cols:
    print(f"  {col}: {sorted(df_w[col].dropna().unique(), key=str)[:10]}")

df_w.to_csv(DATA_PATH / "wrangled_data.csv", index=False)
print("\nSaved to data/wrangled_data.csv")

Final shape: 38,340 rows × 64 columns

Remaining object/string columns (may need attention):
  dataset: ['DAPHNIE 2023', 'DAPHNIE 2024', 'HSE 2017', 'HSE 2018', 'HSE 2019', 'HSE 2022']
  SurveyWeight: [' ', '0.80', '0.84', '0.90', '0.92', '1.00', '1.01', '1.02', '1.14', '1.54']
  eth_oth: [' ', '-', 'African', 'African british', 'Anglo- indian', 'Arab', 'Asian', 'BERBER', 'Balkan', 'Bhutanese']
  eth_oth2: [' ', 'AMERICAN', 'Ashkenazi Jewish- we need a category', 'British', 'British 91%', 'Chinese', 'ENGLISH', 'English', 'English bread and born', 'Hong Kong']
  emp_daphniepilot: [' ', '-o', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8']
  emp_cat_daphnie: ['Employed', 'Other (Sick/Home/etc)', 'Retired', 'Student', 'Unemployed']
  eth4cat_hse2: ['Asian', 'Black', 'Others', 'White']
  emp_cat_hse2019: ['Employed', 'Other (Sick/Home/etc)', 'Retired', 'Student', 'Unemployed']
  emp_cat_hse: ['Employed', 'Other (Sick/Home/etc)', 'Retired', 'Student', 'Unemployed']
  paVig_daphnie: ['1 to 2

In [21]:
resp_pct = (
    df_w.groupby("dataset")['resp']
    .value_counts(normalize=True, dropna=False)
    .mul(100).round(1)
    .rename("pct")
    .reset_index()
    .pivot(index="resp", columns="dataset", values="pct")
    .fillna(0)
)

resp_n = pd.crosstab(
    df_w["resp"].fillna("(missing)"),
    df_w["dataset"],
    margins=True,
    margins_name="Total",
)

# column-wise proportions (% within each dataset)
resp_pct = resp_n.div(resp_n.loc["Total"]).mul(100).round(1)

print("Counts:")
display(resp_n)

print("\nColumn % (within each dataset):")
display(resp_pct)

Counts:


dataset,DAPHNIE 2023,DAPHNIE 2024,HSE 2017,HSE 2018,HSE 2019,HSE 2022,Total
resp,,,,,,,
0.0,1515,4758,7186,7304,0,6995,27758
1.0,142,479,654,686,0,595,2556
(missing),0,0,0,0,8026,0,8026
Total,1657,5237,7840,7990,8026,7590,38340



Column % (within each dataset):


dataset,DAPHNIE 2023,DAPHNIE 2024,HSE 2017,HSE 2018,HSE 2019,HSE 2022,Total
resp,,,,,,,
0.0,91.4,90.9,91.7,91.4,0.0,92.2,72.4
1.0,8.6,9.1,8.3,8.6,0.0,7.8,6.7
(missing),0.0,0.0,0.0,0.0,100.0,0.0,20.9
Total,100.0,100.0,100.0,100.0,100.0,100.0,100.0
